# 10 Attempt Complexity Features

**Phase 5 E-1 — Behavioral Complexity from Attempt Data**  
Schema version: `complexity_v1`

Computes per-learner x per-task complexity features from attempt and session CSVs.
No SQL text is required — all features are derived from attempt metadata.

## Pipeline

```
Raw CSVs
  attempt_*.csv   <- academy_member_id, task_id, attempt_no, is_correct,
                     error_type, execution_time_ms, attempt_type
  session_*.csv   <- academy_member_id, task_id, batch_code, task_type, learner_group
        |
        v
  [0] Configuration
  [1] Load input CSVs + validate required columns
  [2] Merge session context (batch_code, task_type)
  [3] Compute complexity features per learner x task
  [4] Distribution charts
  [5] Write parquet artifact
  [6] Write manifest JSON
  [7] Validation (7 checks)
```

> WARNING: TECHNICAL VALIDATION ONLY — mock dataset.  
> Final thesis requires >=60 participants with expert-validated labels.

## 0. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)

RAW_DIR      = Path('data/raw')
FEATURES_DIR = Path('data/features')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# Pin overrides -- injected by runner; None = auto-detect newest
ATTEMPT_CSV : str | None = None
SESSION_CSV : str | None = None

SCHEMA_VERSION = 'complexity_v1'

print(f'Schema version : {SCHEMA_VERSION}')
print(f'Raw dir        : {RAW_DIR}')
print(f'Features dir   : {FEATURES_DIR}')

## 1. Load input files

In [ ]:
def newest_matching(pattern: str) -> Path | None:
    files = sorted(RAW_DIR.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    h.update(p.read_bytes())
    return h.hexdigest()[:16]

attempt_path = Path(ATTEMPT_CSV) if ATTEMPT_CSV else newest_matching('attempt_*.csv')
session_path = Path(SESSION_CSV) if SESSION_CSV else newest_matching('session_*.csv')

for label, p in [('attempt', attempt_path), ('session', session_path)]:
    if p is None:
        raise FileNotFoundError(
            f'No {label} CSV found in {RAW_DIR}. '
            f'Run: node scripts/export-research-snapshots.mjs --batch <BATCH_CODE>'
        )
    print(f'{label:10s}: {p}')

attempt_df = pd.read_csv(attempt_path, encoding='utf-8-sig')
session_df = pd.read_csv(session_path, encoding='utf-8-sig')

attempt_df['is_correct']        = pd.to_numeric(attempt_df['is_correct'],        errors='coerce').fillna(0).astype(bool)
attempt_df['attempt_no']        = pd.to_numeric(attempt_df['attempt_no'],        errors='coerce')
attempt_df['execution_time_ms'] = pd.to_numeric(attempt_df['execution_time_ms'], errors='coerce')

print(f'\nattempt rows : {len(attempt_df):,}  |  columns : {attempt_df.shape[1]}')
print(f'session rows : {len(session_df):,}  |  columns : {session_df.shape[1]}')

In [ ]:
REQ_ATTEMPT = {'academy_member_id', 'task_id', 'attempt_no', 'is_correct',
               'error_type', 'execution_time_ms', 'attempt_type'}
REQ_SESSION = {'academy_member_id', 'task_id', 'batch_code', 'task_type', 'learner_group'}

def check_required(df: pd.DataFrame, required: set, name: str) -> None:
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f'{name} missing required columns: {missing}')
    print(f'  {name:12s} required columns: OK')

check_required(attempt_df, REQ_ATTEMPT, 'attempt')
check_required(session_df, REQ_SESSION, 'session')

## 2. Merge session context

In [ ]:
# One session context row per (academy_member_id, task_id)
session_keys = (
    session_df[['academy_member_id', 'task_id', 'batch_code', 'task_type', 'learner_group']]
    .drop_duplicates(subset=['academy_member_id', 'task_id'])
)

merged_df = attempt_df.merge(session_keys, on=['academy_member_id', 'task_id'], how='left')

n_unmatched = int(merged_df['batch_code'].isna().sum())
print(f'Attempt rows          : {len(attempt_df):,}')
print(f'After session merge   : {len(merged_df):,}')
print(f'Unmatched (no session): {n_unmatched}')
if n_unmatched > 0:
    print('  WARNING: some attempts have no matching session row -- batch_code will be null.')

## 3. Compute complexity features per learner x task

| Feature | Description |
|---|---|
| `attempt_count` | Total attempts |
| `first_correct_attempt` | Min `attempt_no` where `is_correct==True`; null if never |
| `correct_ratio` | mean(`is_correct`) |
| `error_type_diversity` | nunique of non-null `error_type` values |
| `avg_exec_ms` | mean(`execution_time_ms`), fillna 0 |
| `max_exec_ms` | max(`execution_time_ms`), fillna 0 |
| `has_error` | any `error_type` non-null |
| `error_streak_max` | max consecutive non-correct attempts |
| `complexity_score` | composite 0-100 |

**complexity_score formula:**
```
attempt_component = min(attempt_count / 10.0, 1.0) * 40
error_component   = min(error_type_diversity / 5.0, 1.0) * 30
timing_component  = min(avg_exec_ms / 5000.0, 1.0) * 30
complexity_score  = attempt_component + error_component + timing_component
```

In [ ]:
def error_streak_max(correct_series: pd.Series) -> int:
    """Maximum run of consecutive non-correct attempts (sorted by attempt_no)."""
    max_s = cur = 0
    for v in correct_series:
        if not bool(v):
            cur += 1
            if cur > max_s:
                max_s = cur
        else:
            cur = 0
    return max_s

In [ ]:
records = []

for (member_id, task_id), grp in merged_df.groupby(['academy_member_id', 'task_id'], sort=False):
    grp_s = grp.sort_values('attempt_no').reset_index(drop=True)

    attempt_count         = len(grp_s)
    correct_rows          = grp_s.loc[grp_s['is_correct'], 'attempt_no']
    first_correct_attempt = float(correct_rows.min()) if len(correct_rows) > 0 else float('nan')
    correct_ratio         = float(grp_s['is_correct'].mean())
    error_type_diversity  = int(grp_s['error_type'].dropna().nunique())
    avg_exec_ms           = float(grp_s['execution_time_ms'].fillna(0).mean())
    max_exec_ms           = float(grp_s['execution_time_ms'].fillna(0).max())
    has_error             = bool(grp_s['error_type'].notna().any())
    streak                = error_streak_max(grp_s['is_correct'])

    attempt_component = min(attempt_count / 10.0, 1.0) * 40.0
    error_component   = min(error_type_diversity / 5.0, 1.0) * 30.0
    timing_component  = min(avg_exec_ms / 5000.0, 1.0) * 30.0
    complexity_score  = attempt_component + error_component + timing_component

    batch_code = grp_s['batch_code'].dropna().iloc[0] if grp_s['batch_code'].notna().any() else None
    task_type  = grp_s['task_type'].dropna().iloc[0]  if grp_s['task_type'].notna().any()  else None

    records.append({
        'academy_member_id':     member_id,
        'task_id':               task_id,
        'batch_code':            batch_code,
        'task_type':             task_type,
        'attempt_count':         attempt_count,
        'first_correct_attempt': first_correct_attempt,
        'correct_ratio':         correct_ratio,
        'error_type_diversity':  error_type_diversity,
        'avg_exec_ms':           avg_exec_ms,
        'max_exec_ms':           max_exec_ms,
        'has_error':             has_error,
        'error_streak_max':      streak,
        'complexity_score':      round(complexity_score, 4),
    })

complexity_df = pd.DataFrame(records)
print(f'Feature rows (learner x task): {len(complexity_df):,}')
print(f'Unique learners               : {complexity_df["academy_member_id"].nunique()}')
print(f'Unique tasks                  : {complexity_df["task_id"].nunique()}')
print(complexity_df.describe().round(3).to_string())

## 4. Distribution charts

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt

reports_dir = Path('../reports/phase5')
reports_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(complexity_df['complexity_score'], bins=20, edgecolor='black', color='steelblue')
axes[0].set_xlabel('complexity_score')
axes[0].set_ylabel('Count')
axes[0].set_title('Complexity Score Distribution')

axes[1].hist(
    complexity_df['attempt_count'],
    bins=max(1, int(complexity_df['attempt_count'].max())),
    edgecolor='black', color='darkorange',
)
axes[1].set_xlabel('attempt_count')
axes[1].set_ylabel('Count')
axes[1].set_title('Attempt Count Distribution')

axes[2].hist(complexity_df['correct_ratio'], bins=20, edgecolor='black', color='seagreen')
axes[2].set_xlabel('correct_ratio')
axes[2].set_ylabel('Count')
axes[2].set_title('Correct Ratio Distribution')

plt.tight_layout()
chart_path = reports_dir / 'nb10_complexity_distributions.png'
plt.savefig(chart_path, dpi=150)
plt.close()
print(f'Chart saved: {chart_path}')

## 5. Write output parquet

In [ ]:
OUTPUT_COLS = [
    'academy_member_id', 'task_id', 'batch_code', 'task_type',
    'attempt_count', 'first_correct_attempt', 'correct_ratio',
    'error_type_diversity', 'avg_exec_ms', 'max_exec_ms',
    'has_error', 'error_streak_max', 'complexity_score',
]

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
parquet_path = FEATURES_DIR / 'sql_complexity_v1.parquet'
complexity_df[OUTPUT_COLS].to_parquet(parquet_path, index=False)
print(f'Parquet saved: {parquet_path}  ({len(complexity_df)} rows)')
print(complexity_df[OUTPUT_COLS].head().to_string())

## 6. Write manifest JSON

In [ ]:
manifest = {
    'schema_version': SCHEMA_VERSION,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_files': {
        'attempt_csv': str(attempt_path),
        'attempt_sha': sha256_file(attempt_path),
        'session_csv': str(session_path),
        'session_sha': sha256_file(session_path),
    },
    'parameters': {
        'complexity_score_formula': {
            'attempt_component': 'min(attempt_count / 10.0, 1.0) * 40',
            'error_component':   'min(error_type_diversity / 5.0, 1.0) * 30',
            'timing_component':  'min(avg_exec_ms / 5000.0, 1.0) * 30',
            'total':             'attempt_component + error_component + timing_component',
        },
        'thresholds': {
            'attempt_count_max_for_normalisation': 10,
            'error_diversity_max_for_normalisation': 5,
            'exec_ms_max_for_normalisation': 5000,
        },
    },
    'dataset_stats': {
        'total_rows':            int(len(complexity_df)),
        'unique_learners':       int(complexity_df['academy_member_id'].nunique()),
        'unique_tasks':          int(complexity_df['task_id'].nunique()),
        'complexity_score_mean': round(float(complexity_df['complexity_score'].mean()), 4),
        'complexity_score_min':  round(float(complexity_df['complexity_score'].min()),  4),
        'complexity_score_max':  round(float(complexity_df['complexity_score'].max()),  4),
        'avg_attempt_count':     round(float(complexity_df['attempt_count'].mean()),    4),
        'has_error_pct':         round(float(complexity_df['has_error'].mean() * 100),  2),
        'missing_session_rows':  int(complexity_df['batch_code'].isna().sum()),
    },
    'data_warning': (
        'TECHNICAL VALIDATION ONLY -- mock dataset. '
        'Final thesis requires >=60 participants with expert-validated labels.'
    ),
    'artifacts': {
        'parquet': str(parquet_path),
        'chart':   str(chart_path),
    },
}

manifest_path = FEATURES_DIR / 'complexity_manifest_v1.json'
manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
print(f'Manifest saved: {manifest_path}')
print(json.dumps(manifest['dataset_stats'], indent=2))

## 7. Validation -- 7 checks

In [ ]:
checks = []

def chk(name: str, passed: bool, detail: str = '') -> None:
    checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})

chk('Parquet file exists',
    parquet_path.exists(),
    str(parquet_path))

chk('Row count > 0',
    len(complexity_df) > 0,
    f'{len(complexity_df)} rows')

chk('No null academy_member_id',
    complexity_df['academy_member_id'].notna().all(),
    f'{complexity_df["academy_member_id"].isna().sum()} nulls')

chk('complexity_score in [0, 100]',
    (complexity_df['complexity_score'] >= 0).all() and (complexity_df['complexity_score'] <= 100).all(),
    f'min={complexity_df["complexity_score"].min():.4f}  max={complexity_df["complexity_score"].max():.4f}')

chk('attempt_count >= 1 for all rows',
    (complexity_df['attempt_count'] >= 1).all(),
    f'min={complexity_df["attempt_count"].min()}')

chk('correct_ratio in [0.0, 1.0]',
    (complexity_df['correct_ratio'] >= 0.0).all() and (complexity_df['correct_ratio'] <= 1.0).all(),
    f'min={complexity_df["correct_ratio"].min():.4f}  max={complexity_df["correct_ratio"].max():.4f}')

chk('error_type_diversity >= 0',
    (complexity_df['error_type_diversity'] >= 0).all(),
    f'min={complexity_df["error_type_diversity"].min()}')

result_df = pd.DataFrame(checks)
n_fail    = int((result_df['result'] == 'FAIL').sum())

print('\n-- NB10 Validation Summary ---------------------------------------------------')
print(result_df.to_string(index=False))
print(f'\n{len(checks) - n_fail}/{len(checks)} checks passed')

if n_fail > 0:
    raise RuntimeError(f'NB10 validation FAILED -- {n_fail} check(s) did not pass.')

print('\nNB10 COMPLETE -- artifacts ready for NB11 (error clustering) and NB12 (embeddings).')
print(f'\n   Parquet : {parquet_path}')
print(f'   Manifest: {manifest_path}')
print('\nWARNING: TECHNICAL VALIDATION ONLY.')
print(f'   Rows={len(complexity_df)}  Learners={complexity_df["academy_member_id"].nunique()}  '
      f'Tasks={complexity_df["task_id"].nunique()}')